In [ ]:
from ctd_toolkit.analysis import Analysis
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import pandas as pd
import cmocean
import numpy as np
plt.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern"]})

In [ ]:
variable = 'Temperature'
# path = f'C:/Users/m1_gui01/Desktop/postdoc/oceano/{variable.lower()}/'
path = f'C:/Users/m1_gui01/Desktop/postdoc/oceano/TEMP_-180.0_180.nc'
if variable == 'Salinity' :
    cmap = cmocean.cm.haline
    unit = '(g/kg)'
elif variable == 'Temperature':
    cmap = cmocean.cm.thermal
    unit = '(°C)'

In [ ]:
time_range = (pd.Timestamp(2004,1,1), pd.Timestamp(2025,12,31))

In [ ]:
inst = Analysis(grids_path = path, lat_range = (-73, -40), lon_range = (-180, 180), depth_range = (0,700), time_range = time_range)

In [ ]:
vmin = np.nanpercentile(inst.subset['values'], 1)
vmax = np.nanpercentile(inst.subset['values'], 99)

fig, ax = plt.subplots(
    2, 2,
    figsize=(18,11),
    subplot_kw={'projection': ccrs.PlateCarree()},
    gridspec_kw={'wspace': 0.05}) #ccrs.PlateCarree()
ax = ax.flatten()

from scipy.ndimage import generic_filter
def nan_filter_median(values):
    center = values[len(values) // 2]
    if np.isnan(center):
        valid = values[~np.isnan(values)]
        return np.median(valid) if len(valid) > 0 else np.nan
    return center

for i, (years, season) in enumerate(zip([[6,7,8], [9,10,11],[12,1,2],[3,4,5]], ['Winter','Spring','Summer','Fall'])) :
    data = inst.subset.sel(time=inst.subset['time.month'].isin(list(years)))
    data = data.mean(dim=['time', 'depth'])['values'].values
    ax[i].set_extent([-180, 180, -71, -40])
    ax[i].coastlines(linewidth=0.5, zorder=0)
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.LAND, facecolor='lightgrey')
    latitude = np.append(inst.subset['latitude'].values, np.max(inst.subset['latitude'].values) + 0.1)
    longitude = np.append(inst.subset['longitude'].values, np.max(inst.subset['longitude'].values) + 0.1)
    lon2d, lat2d = np.meshgrid(longitude, latitude)
    img = ax[i].imshow(
        generic_filter(data, nan_filter_median, size=3, mode='reflect')[::-1],
        origin='upper',
        extent=[-180, 180, -73, -40],
        transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal,
        vmin=vmin, vmax=vmax,
        interpolation='bilinear',
        interpolation_stage='rgba')
    ax[i].set_title(season)
fig.subplots_adjust(hspace=0.05, top=0.95, right = 0.85, bottom=0.05)
cbar_ax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
fig.colorbar(img, cax=cbar_ax, orientation="vertical", label=f"{variable} {unit}", shrink=0.5)
for a in ax.flat:
    a.set_aspect('auto')
fig.show()
# fig.savefig(f'C:/Users/m1_gui01/Desktop/postdoc/oceano/{variable.lower()}_maps.pdf')

In [ ]:
## PLOT SEASONAL AND TEMPORAL EVOLUTION

inst = Analysis(grids_path = path, lat_range = (-60, -40), lon_range = (30, 120), depth_range = (100, 700), time_range = time_range)
data = inst.spatial_mean()
data = data.mean(dim = 'depth')
data_time = data['time']
fig, ax = plt.subplots(2,1, figsize = (10, 12))
ax[0].plot(data_time, data['values'].values)
for months, season, color in zip([[6,7,8], [9,10,11],[12,1,2],[3,4,5]], ['Winter','Spring','Summer','Fall'], ['navy', 'limegreen', 'gold', 'darkorange']) :
    _season = inst.subset.sel(time=inst.subset['time.month'].isin(months))
    time_season = _season['time'].groupby('time.year').mean()
    season_group = _season.groupby('time.year')
    mean = season_group.mean(dim=['time', 'depth', 'latitude', 'longitude'])['values'].values
    # q25 = season_group.quantile(0.25, dim=['time', 'depth', 'latitude', 'longitude'])['values'].values
    # q75 = season_group.quantile(0.75, dim=['time', 'depth', 'latitude', 'longitude'])['values'].values
    #ax[1].fill_between(time_season, q25, q75, alpha=0.1, color=color)
    ax[1].plot(time_season, _season.groupby('time.year').mean(dim = ['time','depth','latitude','longitude'])["values"].values, '-o', label=season, color=color)
ax[1].legend()
ax[0].grid(alpha = 0.3)
ax[1].grid(alpha = 0.3)
ax[1].set_ylabel(f"{variable} {unit}")
ax[0].set_ylabel(f"{variable} {unit}")
fig.tight_layout()
# fig.savefig(f'C:/Users/m1_gui01/Desktop/postdoc/oceano/{variable.lower()}_evolution_60S.pdf')

In [ ]:
inst = Analysis(grids_path = path, lat_range = (-73, -50), lon_range = (40, 120), depth_range = (300, 700), time_range = time_range)
_season = inst.subset.sel(time=inst.subset['time.month'].isin([2,3,4,5,6]))
time_season = _season['time'].groupby('time.year').mean()
season_group = _season.groupby('time.year')
mean = season_group.mean(dim=['time', 'depth', 'latitude', 'longitude'])['values'].values


In [ ]:
results = season_group.mean(dim = ['time', 'depth', 'longitude'])

In [ ]:
results

In [ ]:
plt.figure(figsize = (10, 20))
for i in range(22) :
    plt.plot(results['values'].values[i, :], results['latitude'].values, '-o', c = cmap(i/22))
plt.legend()

In [ ]:
## PLOT DEPTH BINNED EVOLUTION

fig, ax = plt.subplots(2,2, figsize = (15, 10))
ax = ax.flatten()

for depth_min, depth_max in zip([0,50,100,150,200,250,300,400,500], [50,100,150,200,250,300,400,500,700]):
    inst = Analysis(grids_path = path, lat_range = (-73, -60), lon_range = (30, 120), depth_range = (depth_min,depth_max), time_range = time_range)
    data = inst.spatial_mean()
    data = data.mean(dim = 'depth')
    data_time = data['time']

    for i, (months, season, color) in enumerate(zip([[6,7,8], [9,10,11],[12,1,2],[3,4,5]], ['Winter','Spring','Summer','Fall'], ['navy', 'limegreen', 'gold', 'darkorange'])) :
        _season = inst.subset.sel(time=inst.subset['time.month'].isin(months))
        time_season = _season['time'].groupby('time.year').mean()
        season_group = _season.groupby('time.year')
        mean = season_group.mean(dim=['time', 'depth', 'latitude', 'longitude'])['values'].values
        ax[i].plot(time_season, _season.groupby('time.year').mean(dim = ['time','depth','latitude','longitude'])["values"].values,
                   '-o',
                    markeredgecolor='black',
                    markeredgewidth=0.3,
                   label=f'{depth_min}-{depth_max} m',
                   color=cmocean.cm.ice(abs(depth_min-700)/700))
ax[0].legend()
ax[0].grid(alpha = 0.3)
ax[1].grid(alpha = 0.3)
ax[2].grid(alpha = 0.3)
ax[3].grid(alpha = 0.3)
ax[0].set_title('Winter')
ax[1].set_title('Spring')
ax[2].set_title('Summer')
ax[3].set_title('Fall')
ax[0].set_ylabel(f"{variable} {unit}")
ax[2].set_ylabel(f"{variable} {unit}")
fig.tight_layout()
fig.savefig(f'C:/Users/m1_gui01/Desktop/postdoc/oceano/{variable.lower()}_depth_60S.pdf')

In [ ]:
def gradient_skip_nan_3d(data, t):
    # data shape: (time, lat, lon)
    nt, ny, nx = data.shape
    flat = data.reshape(nt, -1)        # (time, npoints)
    grad = np.full_like(flat, np.nan)
    for j in range(flat.shape[1]):
        arr = flat[:, j]
        valid = ~np.isnan(arr)
        if valid.sum() > 1:
            idx = np.where(valid)[0]
            grad[idx[1:], j] = (arr[idx[1:]] - arr[idx[:-1]]) / (t[idx[1:]] - t[idx[:-1]])
    return grad.reshape(nt, ny, nx)